# PAN Number Extraction

In [9]:
!pip install imutils pytesseract opencv-python pillow xlsxwriter

In [10]:
from PIL import Image
import re
import numpy as np
from imutils import paths
import os
import pytesseract
import cv2
import imutils
import glob
import xlsxwriter

In [11]:
# Set up images folder path
images_path = r"C:\Users\Shubham Yadav\Documents\GitHub\PanNumberExtraction-main"

In [12]:
workbook = xlsxwriter.Workbook('/content/example.xlsx')
worksheet = workbook.add_worksheet("My sheet")

rectKernel = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
sqKernel = cv2.getStructuringElement(cv2.MORPH_RECT, (21, 21))

row = 0
col = 0
i = 0

for imagePath in paths.list_images(images_path):
    i += 1
    print("Processing:", i)

    image = cv2.imread(imagePath)
    image = imutils.resize(image, height=600)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, rectKernel)

    gradX = cv2.Sobel(blackhat, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=-1)
    gradX = np.absolute(gradX)

    (minVal, maxVal) = (np.min(gradX), np.max(gradX))
    gradX = (255 * ((gradX - minVal) / (maxVal - minVal))).astype("uint8")

    gradX = cv2.morphologyEx(gradX, cv2.MORPH_CLOSE, rectKernel)
    thresh = cv2.threshold(gradX, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]

    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, sqKernel)
    thresh = cv2.erode(thresh, None, iterations=4)

    p = int(image.shape[1] * 0.05)
    thresh[:, 0:p] = 0
    thresh[:, image.shape[1] - p:] = 0

    cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL,
                           cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)
    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)

    for c in cnts:
        (x, y, w, h) = cv2.boundingRect(c)

        pX = int((x + w) * 0.03)
        pY = int((y + h) * 0.03)

        (x, y) = (x - pX, y - pY)
        (w, h) = (w + (pX * 2), h + (pY * 2))

        roi = image[y:y + h, x:x + w].copy()

        filename = "temp.jpg"
        cv2.imwrite(filename, roi)

        text = pytesseract.image_to_string(Image.open(filename), config='--psm 6')
        os.remove(filename)

        pattern = re.compile(r'[A-Z]{5}\d{4}[A-Z]')
        result = ''.join(pattern.findall(text))

        print("Detected:", result)

        if result:
            worksheet.write(row, col, i)
            worksheet.write(row, col + 1, result)

    row += 1

workbook.close()
print("Done. Excel saved at /content/example.xlsx")

Done. Excel saved at /content/example.xlsx
